## ⚠️ Постоянное хранилище на Kaggle — прочитай перед запуском

Все файлы устанавливаются в **`/kaggle/working/sd-persistent/`** (venv, WebUI, модели).

**Первая сессия** → запусти ячейки 0–3 в обычном режиме. По завершении:
1. Нажми **Save Version** (вкладка Data → Output) — Kaggle сохранит `/kaggle/working/` как выходной датасет.
2. Перейди в **Data → Output → New Dataset** и сохрани его как постоянный датасет (например, `sd-persistent`).

**Последующие сессии** → добавь сохранённый датасет как **Input** в настройках ноутбука.
Скрипт автоматически восстановит установку из `/kaggle/input/<датасет>/` перед запуском —
долгая установка будет пропущена.

Ячейки **«Инструменты persistance»** внизу позволяют проверить статус и получить инструкции по сохранению.

In [ ]:
# @title 0. Setup Environment
from pathlib import Path
import glob, os

lang, branch = 'ru', 'main'

# On Kaggle, scripts live inside the persistent dir so they survive restores
if 'KAGGLE_URL_BASE' in os.environ:
    scripts_dir = Path('/kaggle/working/sd-persistent/ANXETY/scripts')
else:
    scripts_dir = Path.home() / 'ANXETY' / 'scripts'

setup_py = scripts_dir / 'setup.py'
os.makedirs(setup_py.parent, exist_ok=True)

if not setup_py.exists():
    # 1) Try to copy from an Input dataset (user uploaded the zip as a dataset)
    _candidates = glob.glob('/kaggle/input/*/scripts/setup.py')
    if _candidates:
        import shutil
        _src = sorted(_candidates)[-1]
        # Copy entire scripts folder from the input dataset
        _scripts_src = Path(_src).parent
        shutil.copytree(str(_scripts_src), str(scripts_dir), dirs_exist_ok=True)
        print(f'Copied scripts from Input dataset: {_scripts_src}')
    else:
        # 2) Fallback: download from GitHub (first ever run, no dataset uploaded)
        !curl -sLo {setup_py} https://raw.githubusercontent.com/anxety-solo/sdAIgen/{branch}/scripts/setup.py
        print('Downloaded setup.py from GitHub (no Input dataset found)')
else:
    print('Using existing setup.py from persistent storage')

%run $setup_py --lang=$lang --branch=$branch


---
## 1. Виджеты 🔽

In [ ]:
# Выбор модели, vae, control-net и многое другое.
%run $scripts_dir/$lang/widgets-{lang}.py

## 2. Загрузка 🔽

In [ ]:
# Загрузка библиотек, репозиториев, моделей и многого другого.
%run $scripts_dir/$lang/downloading-{lang}.py

## 3. Запуск 🔽

In [ ]:
# Запуск WebUI | (используй -t [m|e|d] для теггера: merged/e621/danbooru)
%run $scripts_dir/launch.py -t d

---
# Утилиты

In [ ]:
# Запуск AutoCleaner.
%run $scripts_dir/auto-cleaner.py

---
## 💾 Инструменты постоянного хранилища (Kaggle)

In [ ]:
# Показать статус установки и информацию о восстановлении из Input-датасета
%run $scripts_dir/kaggle_persist.py --status

In [ ]:
# Восстановить вручную из Input-датасета (если автовосстановление не сработало)
%run $scripts_dir/kaggle_persist.py --restore

In [ ]:
# Инструкции по сохранению установки как Kaggle Dataset
%run $scripts_dir/kaggle_persist.py --save-info